In [1]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

In [2]:
import polars as pl
from config.rutas import RUTA_DATA_RAW

In [22]:
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

In [4]:
lf = pl.scan_csv(RUTA_DATA_RAW / "endireh_2021.csv")

In [7]:
CUANTITATIVAS = [
    "edad_primer_union",
    "num_hijos",
    "ingreso_pareja"
]

In [8]:
def limpiar_cuantitativa(lf, variable):
    """Elimina los valores nulos y los códigos especiales de las variables cuantitativas."""
    df = lf.filter(
        pl.col(variable).is_not_null()
        & pl.col("factor_expansion").is_not_null()
        & (pl.col("factor_expansion") > 0)
    )
    if variable == "edad_primer_union":
        df = df.filter(
            ~pl.col(variable).is_in([98, 99])
        )
    elif variable == "ingreso_pareja":
        df = df.filter(
            ~pl.col(variable).is_in([999998, 999999])
        )
    return df

In [25]:
for variable in CUANTITATIVAS:
    df = limpiar_cuantitativa(lf, variable).collect()
    resumen = df.select(
        pl.col(variable).quantile(0.1).alias("P10"),
        pl.col(variable).quantile(0.25).alias("Q1"),
        pl.col(variable).quantile(0.75).alias("Q3"),
        pl.col(variable).quantile(0.9).alias("P90"),
        pl.col(variable).drop_nulls().mode().first().alias("Moda"),
        pl.col(variable).median().alias("Mediana"),
        pl.col(variable).mean().alias("Media"),
    )
    valores = df[variable].to_numpy()
    pesos = df["factor_expansion"].to_numpy()
    media_ponderada = np.average(valores, weights=pesos)
    resumen = resumen.with_columns(pl.lit(media_ponderada).alias("Media ponderada"))
    print(f"Resumen de {variable}:")
    print(resumen)

Resumen de edad_primer_union:
shape: (1, 8)
┌─────┬─────┬──────┬──────┬──────┬─────────┬───────────┬─────────────────┐
│ P10 ┆ Q1  ┆ Q3   ┆ P90  ┆ Moda ┆ Mediana ┆ Media     ┆ Media ponderada │
│ --- ┆ --- ┆ ---  ┆ ---  ┆ ---  ┆ ---     ┆ ---       ┆ ---             │
│ f64 ┆ f64 ┆ f64  ┆ f64  ┆ f64  ┆ f64     ┆ f64       ┆ f64             │
╞═════╪═════╪══════╪══════╪══════╪═════════╪═══════════╪═════════════════╡
│ 0.0 ┆ 2.0 ┆ 15.0 ┆ 25.0 ┆ 0.0  ┆ 6.0     ┆ 10.026312 ┆ 9.700189        │
└─────┴─────┴──────┴──────┴──────┴─────────┴───────────┴─────────────────┘
Resumen de num_hijos:
shape: (1, 8)
┌─────┬─────┬─────┬─────┬──────┬─────────┬──────────┬─────────────────┐
│ P10 ┆ Q1  ┆ Q3  ┆ P90 ┆ Moda ┆ Mediana ┆ Media    ┆ Media ponderada │
│ --- ┆ --- ┆ --- ┆ --- ┆ ---  ┆ ---     ┆ ---      ┆ ---             │
│ f64 ┆ f64 ┆ f64 ┆ f64 ┆ f64  ┆ f64     ┆ f64      ┆ f64             │
╞═════╪═════╪═════╪═════╪══════╪═════════╪══════════╪═════════════════╡
│ 1.0 ┆ 1.0 ┆ 3.0 ┆ 4.0 ┆ 3.0  ┆ 3.